In [1]:
import pickle
import os

import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from dataset.dataset import load_data
from softmax_CBM.dataset.dataset import load_data as load_data_softmax
from dataset.utils import *
import matplotlib.pyplot as plt

from models.model import *

torch.set_printoptions(sci_mode=False)

import lrp
from models.lrp_converter import convert

from PIL import Image

/home/jack/anaconda3/envs/playing_cards/lib/python3.10/site-packages/torch/cuda/__init__.py:83: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at  ../c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


In [2]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("Device:", device, torch.cuda.get_device_name(0))
else:
    print("Device:", device)

Device: cpu


In [ ]:
# single playing card dataset has 17 concepts
# multi card dataset has 52 concepts



N_CONCEPT = 17
N_CLASSES = 52
val_loader = load_data("./dataset/playing_cards/val.pkl", 16, dataset_root="./dataset/playing_cards", return_orig=True)

concept_index_to_string = IndexToString("./dataset/playing_cards/attributes.txt")


"""
N_CONCEPT = 52
N_CLASSES = 1
val_loader = load_data("./dataset/three_card_poker/val.pkl", 16, dataset_root="./dataset/three_card_poker", has_class=False, return_orig=True)

concept_index_to_string = IndexToString("./dataset/three_card_poker/concepts.txt")
"""

XtoC = PlayingCardNet(num_classes=N_CONCEPT)
XtoC.load_state_dict(torch.load("./saves/single_XtoC_converted.pth"))
XtoC.to(device)
XtoC.eval()

print("model loaded")

In [3]:
##### https://github.com/fhvilshoj/TorchLRP/blob/74253a1be05f0be0b7c535736023408670443b6e/examples/visualization.py#L60
def heatmap(X, cmap_name="seismic"):
	cmap = plt.cm.get_cmap(cmap_name)

	if X.shape[1] in [1, 3]:  # move channel index to end + convert to np array
		X = X.permute(0, 2, 3, 1).detach().cpu().numpy()
	if isinstance(X, torch.Tensor):  # convert tensor to np array
		X = X.detach().cpu().numpy()

	shape = X.shape
	tmp = X.sum(axis=-1) # Reduce channel axis

	tmp = project(tmp, output_range=(0, 255)).astype(int)
	tmp = cmap(tmp.flatten())[:, :3].T
	tmp = tmp.T

	shape = list(shape)
	shape[-1] = 3
	return tmp.reshape(shape).astype(np.float32)


def project(X, output_range=(0, 1)):
	absmax = np.abs(X).max(axis=tuple(range(1, len(X.shape))), keepdims=True)
	X /= absmax + (absmax == 0).astype(float)
	X = (X+1) / 2. # range [0, 1]
	X = output_range[0] + X * (output_range[1] - output_range[0]) # range [x, y]
	return X

In [ ]:
XtoC = convert(XtoC).to(device)

print(XtoC)

## Show true concepts and classification + predicted concepts and classification

In [ ]:
imageId = 7

images, labels, concepts, original_imgs = next(iter(val_loader))
image, label = images[imageId].to(device), labels[imageId].to(device)

# print / show input
print("True image size (channels, width, height):", image.size())
print("True  image concepts:", concepts[imageId])

plt.xticks([])
plt.yticks([])
plt.imshow(original_imgs[imageId].squeeze().permute(1, 2, 0))
plt.show()

plt.xticks([])
plt.yticks([])
plt.imshow(images[imageId].squeeze().permute(1, 2, 0))
plt.show()

In [ ]:
image, label = images[imageId].to(device), labels[imageId].to(device)
image = image.unsqueeze(0)
image.requires_grad_(True)
image.grad = None

# get model prediction and LRP map for classification
prediction = XtoC(image)

pred_concepts_readable = torch.nn.Sigmoid()(prediction)
print("Predicted concepts:", pred_concepts_readable)

# print concept as string if prediction is >= 0.5
for idx, i in enumerate(pred_concepts_readable[0]):
    if i.item() >= 0.5:
        print(f"id: {idx},", f"concept name: {concept_index_to_string(idx)},", f"sig value: {i.item()}")

In [ ]:
input_image = Image.open("./dataset/playing_cards/cropped/standard_new/10S.png").convert('RGB')
preprocess = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.5, 0.5, 0.5], std = [2, 2, 2])
])
input_tensor = preprocess(input_image)
input_batch = input_tensor.unsqueeze(0) # create a mini-batch as expected by the model

input_batch = input_batch.to(device)

input_batch.requires_grad_(True)
input_batch.grad = None
prediction = XtoC(input_batch)

pred_concepts_readable = torch.nn.Sigmoid()(prediction)
print("Predicted concepts:", pred_concepts_readable)

# print concept as string if prediction is >= 0.5
for idx, i in enumerate(pred_concepts_readable[0]):
    if i.item() >= 0.5:
        print(f"id: {idx},", f"concept name: {concept_index_to_string(idx)},", f"sig value: {i.item()}")

## Generate LRP saliency maps for concepts using a single rule

In [ ]:
image, label = images[imageId].to(device), labels[imageId].to(device)
#image, label = images.to(device), labels.to(device)
image = image.unsqueeze(0)
image.requires_grad_(True)
image.grad = None  # Reset gradient

#pred_concepts = XtoC.forward(image, explain=True, rule="alpha1beta0")
pred_concepts = XtoC.forward(image, explain=True, rule="alpha2beta1")
#pred_concepts = XtoC.forward(image, explain=True, rule="epsilon")
#pred_concepts = XtoC.forward(image, explain=True, rule="gradient")

# get saliecy maps for each concept
for i in range(pred_concepts.size(1)):
    filter_out = torch.zeros_like(pred_concepts)
    filter_out[:,i] += 1

    # Get the gradient of each input
    image_gradient = torch.autograd.grad(
        pred_concepts,
        image,
        grad_outputs=filter_out,
        retain_graph=True)[0]


    attr = heatmap(image_gradient, cmap_name='seismic')
    
    print(concept_index_to_string(i))

    plt.xticks([])
    plt.yticks([])
    #plt.title(label=concept_index_to_string(i), fontsize=15)
    plt.imshow(attr.squeeze(), cmap='seismic')
    #plt.savefig(f'temp/{i}.png')
    plt.show()

    plt.close()

In [ ]:
image = Image.open("./dataset/playing_cards/cropped/standard_new/10S.png").convert('RGB')
preprocess = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.5, 0.5, 0.5], std = [2, 2, 2])
])
image = preprocess(image)
image = image.to(device)
image = image.unsqueeze(0) # create a mini-batch as expected by the model
image.requires_grad_(True)
image.grad = None  # Reset gradient

#pred_concepts = XtoC.forward(image, explain=True, rule="alpha1beta0")
pred_concepts = XtoC.forward(image, explain=True, rule="alpha2beta1")
#pred_concepts = XtoC.forward(image, explain=True, rule="epsilon")
#pred_concepts = XtoC.forward(image, explain=True, rule="gradient")

# get saliecy maps for each concept
for i in range(pred_concepts.size(1)):
    filter_out = torch.zeros_like(pred_concepts)
    filter_out[:,i] += 1

    # Get the gradient of each input
    image_gradient = torch.autograd.grad(
        pred_concepts,
        image,
        grad_outputs=filter_out,
        retain_graph=True)[0]


    attr = heatmap(image_gradient, cmap_name='seismic')
    
    print(concept_index_to_string(i))

    plt.xticks([])
    plt.yticks([])
    #plt.title(label=concept_index_to_string(i), fontsize=15)
    plt.imshow(attr.squeeze(), cmap='seismic')
    #plt.savefig(f'temp/{i}.png')
    plt.show()

    plt.close()

## Generate LRP saliency maps for concepts using composite rules

In [ ]:
XtoC_model_1 = []
XtoC_model_2 = []
XtoC_model_3 = []

for idx, m in enumerate(XtoC.children()):
    if idx < 8:
        XtoC_model_1.append(m)
    elif idx < 15:
        XtoC_model_2.append(m)
    else:
        XtoC_model_3.append(m)

XtoC_model_1 = lrp.Sequential(*XtoC_model_1)
XtoC_model_2 = lrp.Sequential(*XtoC_model_2)
XtoC_model_3 = lrp.Sequential(*XtoC_model_3)

XtoC_model_1.to(device)
XtoC_model_2.to(device)
XtoC_model_3.to(device)

print(XtoC_model_1)
print(XtoC_model_2)
print(XtoC_model_3)

In [ ]:
image, label = images[imageId].to(device), labels[imageId].to(device)
#image, label = images.to(device), labels.to(device)
image = image.unsqueeze(0)
image.requires_grad_(True)
image.grad = None  # Reset gradient

output_1 = XtoC_model_1.forward(image, explain=True, rule="alpha1beta0")
output_2 = XtoC_model_2.forward(output_1, explain=True, rule="epsilon")
pred_concepts = XtoC_model_3.forward(output_2, explain=True, rule="gradient")

"""
# get saliecy maps for a subset of concept (first 15)
for i in range(15):
    filter_out = torch.zeros_like(pred_concepts)
    filter_out[:,i] += 1

    # Get the gradient of each input
    image_gradient = torch.autograd.grad(
        pred_concepts,
        image,
        grad_outputs=filter_out,
        retain_graph=True)[0]


    attr = heatmap(image_gradient, cmap_name='seismic')
    
    print(concept_index_to_string(i))

    plt.xticks([])
    plt.yticks([])
    #plt.title(label=concept_index_to_string(i), fontsize=15)
    plt.imshow(attr.squeeze(), cmap='seismic')
    #plt.savefig(f'temp/{i}.png')
    plt.show()

    plt.close()
"""
    
    
# get saliecy maps for top 5 concepts
pred_precent_concepts = []
for idx, i in enumerate(pred_concepts_readable[0]):
    if i.item() >= 0.5:
        pred_precent_concepts.append((idx, i.item()))

pred_precent_concepts = sorted(pred_precent_concepts, key=lambda x: -x[1])

for i in range(pred_concepts.size(1)):
    filter_out = torch.zeros_like(pred_concepts)
    filter_out[:,i] += 1

    # Get the gradient of each input
    image_gradient = torch.autograd.grad(
        pred_concepts,
        image,
        grad_outputs=filter_out,
        retain_graph=True)[0]


    attr = heatmap(image_gradient, cmap_name='seismic')
    
    print(concept_index_to_string(i))

    plt.xticks([])
    plt.yticks([])
    #plt.title(label=concept_index_to_string(i), fontsize=15)
    plt.imshow(attr.squeeze(), cmap='seismic')
    #plt.savefig(f'temp/{i[0]}.png')
    plt.show()

    plt.close() 


# multi card input model


In [ ]:
NUM_IMG_OUTPUT = 3
USE_SIG = True

In [ ]:
# single playing card dataset has 17 concepts
# multi card dataset has 52 concepts or 17 if using suit and value



N_CONCEPT = 17
N_CLASSES = 6
#val_loader = load_data("./dataset/individual_three_card_poker/val.pkl", 16, num_concepts=N_CONCEPT, dataset_root="./dataset/individual_three_card_poker", return_orig=True, multi_image=True, shuffle=True)
val_loader = load_data_softmax("./softmax_CBM/dataset/individual_three_card_poker/val.pkl", 16, num_concepts=N_CONCEPT, dataset_root="./softmax_CBM/dataset/individual_three_card_poker", return_orig=True, multi_image=True, shuffle=True)


concept_index_to_string = IndexToString("./dataset/individual_three_card_poker/card_concepts.txt")
class_index_to_string = IndexToString("./dataset/individual_three_card_poker/hand_classes.txt")

XtoCtoY = End2EndModel(num_classes=N_CLASSES, num_concepts=N_CONCEPT, num_images_per_output=NUM_IMG_OUTPUT, use_sigmoid=USE_SIG, use_modulelist=False)
XtoCtoY.load_state_dict(torch.load("./saves/softmax_XtoCtoY_best.pth", map_location=torch.device('cpu')))
XtoCtoY.to(device)
XtoCtoY.eval()

print("model loaded")

In [ ]:
XtoC = convert(XtoCtoY.x_to_c_model).to(device)
CtoY = convert(XtoCtoY.c_to_y_model).to(device)

print(XtoC)
print(CtoY)

In [ ]:
imageId = 7

images, labels, concepts, original_imgs = next(iter(val_loader))
image, label = images[imageId].to(device), labels[imageId].to(device)

concepts[imageId] = concepts[imageId].view(3, -1)

# print / show input
print("True image size (channels, width, height):", image.size())
print("True  image concepts:", concepts[imageId])


for i in range(images.size(1)):

    plt.xticks([])
    plt.yticks([])
    plt.imshow(original_imgs[imageId][i, :, :, :].squeeze().permute(1, 2, 0))
    plt.show()

for i in range(images.size(1)):

    plt.xticks([])
    plt.yticks([])
    plt.imshow(images[imageId][i, :, :, :].squeeze().permute(1, 2, 0))
    plt.show()

In [ ]:
image, label = images[imageId].to(device), labels[imageId].to(device)
image = image.unsqueeze(0)
image.requires_grad_(True)
image.grad = None

# get model prediction and LRP map for classification
prediction = XtoCtoY(image)

concept_out = prediction[0]
task_out = prediction[1]

_, task_out = task_out.max(1)

concept_out = concept_out.view(3, -1)

for i in range(concept_out.size(0)):
    pred_concepts_readable = torch.nn.Sigmoid()(concept_out[i])
    print("Predicted concepts:", pred_concepts_readable)

    # print concept as string if prediction is >= 0.5
    for idx, i in enumerate(pred_concepts_readable):
        if i.item() >= 0.5:
            print(f"id: {idx},", f"concept name: {concept_index_to_string(idx)},", f"sig value: {i.item()}")
        
print("Class out:", class_index_to_string(task_out.item()))

In [ ]:
image, label = images[imageId].to(device), labels[imageId].to(device)
#image, label = images.to(device), labels.to(device)
image = image.unsqueeze(0)

if NUM_IMG_OUTPUT != 1:
    # split batch into first hand card, second hand card etc.
    # Run each card batch seperatly
    split_images = []
    for image_id in range(image.size(1)):
        split_images.append(image[:, image_id, :, : , :])
else:
    split_images = [image]  # make sure input is in same format as is there are multiple inputs

outputs = []
for image in split_images:
    image.requires_grad_(True)
    image.grad = None  # Reset gradient
    outputs.append(XtoC.forward(image, explain=True, rule="alpha1beta0"))
    # alpha1beta0, alpha2beta1, epsilon, gradient
    




for idx, pred_concepts in enumerate(outputs):

    # get saliecy maps for each concept
    for i in range(pred_concepts.size(1)):
        filter_out = torch.zeros_like(pred_concepts)
        filter_out[:,i] += 1

        # Get the gradient of each input
        image_gradient = torch.autograd.grad(
            pred_concepts,
            split_images[idx],
            grad_outputs=filter_out,
            retain_graph=True)[0]


        attr = heatmap(image_gradient, cmap_name='seismic')

        print(concept_index_to_string(i))

        plt.xticks([])
        plt.yticks([])
        #plt.title(label=concept_index_to_string(i), fontsize=15)
        plt.imshow(attr.squeeze(), cmap='seismic')
        #plt.savefig(f'temp/{i}.png')
        plt.show()

        plt.close()

# multi card input model with one image per output


In [4]:
NUM_IMG_OUTPUT = 1
USE_SIG = True

In [6]:
# single playing card dataset has 17 concepts
# multi card dataset has 52 concepts or 17 if using suit and value



N_CONCEPT = 52
N_CLASSES = 6
val_loader = load_data("./dataset/three_card_poker_2/val.pkl", 16, num_concepts=N_CONCEPT, dataset_root="./dataset/three_card_poker_2", return_orig=True, multi_image=False, shuffle=True)


concept_index_to_string = IndexToString("./dataset/three_card_poker_2/concepts.txt")
class_index_to_string = IndexToString("./dataset/three_card_poker_2/hand_classes.txt")

XtoCtoY = End2EndModel(num_classes=N_CLASSES, num_concepts=N_CONCEPT, num_images_per_output=NUM_IMG_OUTPUT, use_sigmoid=USE_SIG, use_modulelist=False)
XtoCtoY.load_state_dict(torch.load("./saves/XtoCtoY_converted.pth", map_location=torch.device('cpu')))
XtoCtoY.to(device)
XtoCtoY.eval()

print("model loaded")

KeyError: 'attribute_label'

In [ ]:
XtoC = convert(XtoCtoY.x_to_c_model).to(device)
CtoY = convert(XtoCtoY.c_to_y_model).to(device)

print(XtoC)
print(CtoY)

In [ ]:
imageId = 7

images, labels, concepts, original_imgs = next(iter(val_loader))
image, label = images[imageId].to(device), labels[imageId].to(device)

concepts[imageId] = concepts[imageId].view(3, -1)

# print / show input
print("True image size (channels, width, height):", image.size())
print("True  image concepts:", concepts[imageId])


for i in range(images.size(1)):

    plt.xticks([])
    plt.yticks([])
    plt.imshow(original_imgs[imageId][i, :, :, :].squeeze().permute(1, 2, 0))
    plt.show()

for i in range(images.size(1)):

    plt.xticks([])
    plt.yticks([])
    plt.imshow(images[imageId][i, :, :, :].squeeze().permute(1, 2, 0))
    plt.show()

In [ ]:
image, label = images[imageId].to(device), labels[imageId].to(device)
image = image.unsqueeze(0)
image.requires_grad_(True)
image.grad = None

# get model prediction and LRP map for classification
prediction = XtoCtoY(image)

concept_out = prediction[0]
task_out = prediction[1]

_, task_out = task_out.max(1)

concept_out = concept_out.view(3, -1)

for i in range(concept_out.size(0)):
    pred_concepts_readable = torch.nn.Sigmoid()(concept_out[i])
    print("Predicted concepts:", pred_concepts_readable)

    # print concept as string if prediction is >= 0.5
    for idx, i in enumerate(pred_concepts_readable):
        if i.item() >= 0.5:
            print(f"id: {idx},", f"concept name: {concept_index_to_string(idx)},", f"sig value: {i.item()}")
        
print("Class out:", class_index_to_string(task_out.item()))

In [ ]:
image, label = images[imageId].to(device), labels[imageId].to(device)
#image, label = images.to(device), labels.to(device)
image = image.unsqueeze(0)

if NUM_IMG_OUTPUT != 1:
    # split batch into first hand card, second hand card etc.
    # Run each card batch seperatly
    split_images = []
    for image_id in range(image.size(1)):
        split_images.append(image[:, image_id, :, : , :])
else:
    split_images = [image]  # make sure input is in same format as is there are multiple inputs

outputs = []
for image in split_images:
    image.requires_grad_(True)
    image.grad = None  # Reset gradient
    outputs.append(XtoC.forward(image, explain=True, rule="alpha1beta0"))
    # alpha1beta0, alpha2beta1, epsilon, gradient
    




for idx, pred_concepts in enumerate(outputs):

    # get saliecy maps for each concept
    for i in range(pred_concepts.size(1)):
        filter_out = torch.zeros_like(pred_concepts)
        filter_out[:,i] += 1

        # Get the gradient of each input
        image_gradient = torch.autograd.grad(
            pred_concepts,
            split_images[idx],
            grad_outputs=filter_out,
            retain_graph=True)[0]


        attr = heatmap(image_gradient, cmap_name='seismic')

        print(concept_index_to_string(i))

        plt.xticks([])
        plt.yticks([])
        #plt.title(label=concept_index_to_string(i), fontsize=15)
        plt.imshow(attr.squeeze(), cmap='seismic')
        #plt.savefig(f'temp/{i}.png')
        plt.show()

        plt.close()